In [ ]:
# | default_exp core

In [ ]:
# | export
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from functools import lru_cache
import pandas as pd
import duckdb
import structlog
from datetime import datetime
import json
import time

log = structlog.get_logger()

In [ ]:
# | export
IMPACT_PANELS = ("IMPACT341", "IMPACT410", "IMPACT468", "IMPACT505")
ACCESS_PANELS = ("ACCESS129", "ACCESS146")

VARIANT_KEY_COLS = [
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
]

MAF_USECOLS = [
    "Hugo_Symbol",
    "Tumor_Sample_Barcode",
    "Mutation_Status",
    "Variant_Classification",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "t_ref_count",
    "t_alt_count",
]


@dataclass
class LabelConfig:
    """Configuration for the ctDNA labeling engine."""

    min_vaf: float = 0.01  # VAF threshold for Possible ctDNA+
    min_variants: int = 1  # Minimum # somatic SNVs passing VAF threshold
    min_fragments: int = 2000  # Min fragments for Depth QC
    access_panels: tuple[str, ...] = ACCESS_PANELS
    impact_panels: tuple[str, ...] = IMPACT_PANELS

    def __repr__(self) -> str:
        return (
            f"LabelConfig(min_vaf={self.min_vaf}, "
            f"min_variants={self.min_variants}, "
            f"panels={self.access_panels})"
        )


@dataclass
class Paths:
    """All input paths for the labeling pipeline."""

    cancer_samplesheet: Path
    healthy_xs1_samplesheet: Path
    healthy_xs2_samplesheet: Path
    cbioportal_dir: Path  # directory containing all 5 cBioPortal files
    krewlyzer_dirs: list[Path] = field(default_factory=list)

    def __post_init__(self):
        """Coerce all path fields from str to Path for safe / operations."""
        self.cancer_samplesheet = Path(self.cancer_samplesheet)
        self.healthy_xs1_samplesheet = Path(self.healthy_xs1_samplesheet)
        self.healthy_xs2_samplesheet = Path(self.healthy_xs2_samplesheet)
        self.cbioportal_dir = Path(self.cbioportal_dir)

        expanded = []
        for d in self.krewlyzer_dirs:
            p = Path(str(d).strip().strip('"').strip("'"))
            if p.is_file() and p.suffix == ".txt":
                with open(p) as f:
                    for line in f:
                        line = line.strip().strip('"').strip("'")
                        if line:
                            path_obj = Path(line)
                            if not path_obj.exists():
                                log.warning("manifest_path_missing", path=str(path_obj))
                            expanded.append(path_obj)
            else:
                if not p.exists():
                    log.warning("krewlyzer_dir_missing", path=str(p))
                expanded.append(p)
        self.krewlyzer_dirs = expanded

    @property
    def maf(self) -> Path:
        return self.cbioportal_dir / "data_mutations_extended.txt"

    @property
    def sv(self) -> Path:
        return self.cbioportal_dir / "data_sv.txt"

    @property
    def cna(self) -> Path:
        return self.cbioportal_dir / "data_CNA.txt"

    @property
    def clinical_sample(self) -> Path:
        return self.cbioportal_dir / "data_clinical_sample.txt"

    @property
    def clinical_patient(self) -> Path:
        return self.cbioportal_dir / "data_clinical_patient.txt"

In [ ]:
# | export
def load_samplesheet(path: str | Path) -> pd.DataFrame:
    """Load a krewlyzer samplesheet CSV."""
    if not Path(path).exists():
        log.error("samplesheet_missing", path=str(path))
        raise FileNotFoundError(f"Samplesheet not found: {path}")

    df = pd.read_csv(path)
    log.info("samplesheet_loaded", path=str(path), n_samples=len(df))
    return df


def get_sample_ids(samplesheet_path: str | Path) -> set[str]:
    """Extract unique sample IDs from a samplesheet."""
    df = load_samplesheet(samplesheet_path)
    if "sample" not in df.columns:
        log.error(
            "invalid_samplesheet", path=str(samplesheet_path), columns=list(df.columns)
        )
        raise KeyError("'sample' column missing in samplesheet")
    return set(df["sample"])

In [ ]:
# | export
@lru_cache(maxsize=1)
def load_maf(path: str | Path) -> pd.DataFrame:
    """Load MAF file with only the columns needed for labeling."""
    if not Path(path).exists():
        log.error("maf_missing", path=str(path))
        raise FileNotFoundError(f"MAF not found: {path}")

    log.info("loading_maf", path=str(path))
    try:
        df = pd.read_csv(
            path,
            sep="\t",
            comment="#",
            usecols=MAF_USECOLS,
            dtype={
                "t_ref_count": "Int64",
                "t_alt_count": "Int64",
                "Start_Position": "Int64",
                "End_Position": "Int64",
            },
        )
        log.info(
            "maf_loaded", n_rows=len(df), n_samples=df["Tumor_Sample_Barcode"].nunique()
        )
        return df
    except Exception as e:
        log.error("maf_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_sv(path: str | Path) -> pd.DataFrame:
    """Load structural variant file."""
    if not Path(path).exists():
        log.error("sv_missing", path=str(path))
        raise FileNotFoundError(f"SV file not found: {path}")

    log.info("loading_sv", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t")
        log.info("sv_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("sv_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_cna(path: str | Path) -> pd.DataFrame:
    """Load discrete CNA matrix (wide format). Index = Hugo_Symbol."""
    if not Path(path).exists():
        log.error("cna_missing", path=str(path))
        raise FileNotFoundError(f"CNA not found: {path}")

    log.info("loading_cna", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", index_col=0)
        log.info("cna_loaded", n_genes=df.shape[0], n_samples=df.shape[1])
        return df
    except Exception as e:
        log.error("cna_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_clinical_sample(path: str | Path) -> pd.DataFrame:
    """Load sample-level clinical data (skips 4-line cBioPortal # header)."""
    if not Path(path).exists():
        log.error("clinical_sample_missing", path=str(path))
        raise FileNotFoundError(f"Clinical sample file not found: {path}")

    log.info("loading_clinical_sample", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", comment="#")
        log.info("clinical_sample_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("clinical_sample_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_clinical_patient(path: str | Path) -> pd.DataFrame:
    """Load patient-level clinical data (skips 4-line cBioPortal # header)."""
    if not Path(path).exists():
        log.error("clinical_patient_missing", path=str(path))
        raise FileNotFoundError(f"Clinical patient file not found: {path}")

    log.info("loading_clinical_patient", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", comment="#")
        log.info("clinical_patient_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("clinical_patient_load_failed", path=str(path), error=str(e))
        raise


# | export
def clear_cbioportal_caches():
    """Explicitly clear all cBioPortal LRU caches to prevent memory leaks in long running processes."""
    load_maf.cache_clear()
    load_clinical_sample.cache_clear()
    load_clinical_patient.cache_clear()
    log.info("cbioportal_caches_cleared")

In [ ]:
# | export
import threading

_thread_local = threading.local()


def get_duckdb_conn() -> duckdb.DuckDBPyConnection:
    """Create a thread-local DuckDB connection with optimal settings.
    Configured natively for 4 threads and 4GB memory to prevent HPC OOMs.
    """
    if not hasattr(_thread_local, "conn"):
        conn = duckdb.connect()
        conn.execute("SET threads TO 4")
        conn.execute("SET memory_limit = '4GB'")
        log.debug("duckdb_conn_initialized", threads=4, memory_limit="4GB")
        _thread_local.conn = conn
    return _thread_local.conn


def discover_available_samples(
    results_dirs: list[Path],
    required_suffix: str = ".metadata.parquet",
) -> dict[str, Path]:
    """Discover which samples have completed krewlyzer processing across multiple input directories."""
    available = {}
    for r_dir in results_dirs:
        results_path = Path(r_dir)
        if not results_path.exists():
            log.warning("results_dir_missing", path=str(results_path))
            continue

        # Auto-detect pipeline wrappers (Nextflow output dirs)
        scan_path = results_path
        if (results_path / "results").is_dir():
            scan_path = results_path / "results"

        for sample_dir in scan_path.iterdir():
            if sample_dir.is_dir():
                marker = sample_dir / f"{sample_dir.name}{required_suffix}"
                if marker.exists():
                    available[sample_dir.name] = scan_path

    log.info("sample_discovery", n_available=len(available), num_dirs=len(results_dirs))
    return available

In [ ]:
# | export
def load_feature_cohort(
    feature_suffix: str,
    results_dirs: list[Path],
    sample_ids: list[str] | set[str] | None = None,
    conn: duckdb.DuckDBPyConnection | None = None,
    chunk_size: int = 500,
) -> pd.DataFrame:
    """Load one feature type across available samples using explicit file list.

    ARCHITECTURE NOTE: We build an explicit file list from discovered samples
    instead of using a glob pattern. This avoids DuckDB scanning thousands of
    directories over network mounts (SFTP/NFS), which causes multi-minute stalls.
    """
    start_time = time.time()

    if not results_dirs:
        log.error("results_dirs_empty")
        raise ValueError("Must provide at least one krewlyzer results directory")

    if conn is None:
        conn = get_duckdb_conn()

    # Step 1: Discover valid samples (fast filesystem ls, no parquet reads)
    log.info("discovering_samples", n_dirs=len(results_dirs))
    available_samples = discover_available_samples(
        results_dirs, required_suffix=".metadata.parquet"
    )

    if sample_ids is not None:
        final_samples = sorted(set(sample_ids).intersection(available_samples.keys()))
        if len(final_samples) < len(set(sample_ids)):
            log.warning(
                "samples_missing_krewlyzer_output",
                requested=len(set(sample_ids)),
                found=len(final_samples),
            )
    else:
        final_samples = sorted(available_samples.keys())

    if not final_samples:
        log.warning("no_samples_available_for_cohort", feature=feature_suffix)
        return pd.DataFrame()

    log.info("building_file_list", n_samples=len(final_samples))

    # Step 2: Build explicit file paths (no glob needed)
    file_paths = []
    missing_files = 0
    for sid in final_samples:
        r_dir = available_samples[sid]
        fp = r_dir / sid / f"{sid}{feature_suffix}"
        if fp.exists():
            file_paths.append(str(fp))
        else:
            missing_files += 1

    if missing_files > 0:
        log.info(
            "feature_files_missing",
            feature=feature_suffix,
            missing=missing_files,
            found=len(file_paths),
        )

    if not file_paths:
        log.warning("no_feature_files_found", feature=feature_suffix)
        return pd.DataFrame()

    log.info("reading_parquet_files", n_files=len(file_paths), chunk_size=chunk_size)

    # Step 3: Read explicit file list (pass Python list directly, no subquery)
    query = """
        SELECT *,
            regexp_extract(filename, '/([^/]+)/[^/]+$', 1) AS sample_id
        FROM read_parquet(
            ?,
            filename=true,
            union_by_name=true,
            hive_partitioning=false
        )
    """

    conn.execute("SET threads=4;")  # Throttle SFTP I/O bursts

    df_list = []
    for i in range(0, len(file_paths), chunk_size):
        chunk = file_paths[i : i + chunk_size]
        max_retries = 3

        for attempt in range(max_retries):
            try:
                df_chunk = conn.execute(query, [chunk]).df()
                df_list.append(df_chunk)
                break
            except Exception as e:
                err_str = str(e)
                if attempt < max_retries - 1:
                    log.warning(
                        "duckdb_io_retry",
                        feature=feature_suffix,
                        attempt=attempt + 1,
                        chunk_start=i,
                        error=err_str,
                    )
                    time.sleep(
                        2**attempt
                    )  # Exponential backoff for transient I/O failures
                else:
                    log.error(
                        "feature_cohort_load_failed",
                        feature=feature_suffix,
                        error=err_str,
                        chunk_start=i,
                    )
                    return pd.DataFrame()  # Permanent failure

    df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

    elapsed = time.time() - start_time
    if df.empty:
        log.warning(
            "empty_result", feature=feature_suffix, reason="no_matching_rows_found"
        )
    else:
        log.info(
            "feature_cohort_loaded",
            feature=feature_suffix,
            n_samples=df["sample_id"].nunique(),
            n_rows=len(df),
            elapsed_sec=round(elapsed, 2),
        )
        log.info(
            "load_complete",
            n_samples=df["sample_id"].nunique(),
            n_rows=len(df),
            elapsed_sec=round(elapsed, 1),
        )
    return df


def load_metadata_cohort(results_dirs: list[Path], sample_ids=None):
    return load_feature_cohort(".metadata.parquet", results_dirs, sample_ids)

In [ ]:
# | export
def load_sample_feature(
    sample_id: str,
    feature_suffix: str,
    results_dir: str | Path,
) -> pd.DataFrame:
    """Load a single parquet feature file for one sample. (Useful for single-sample logic)"""
    path = Path(results_dir) / sample_id / f"{sample_id}{feature_suffix}"
    if not path.exists():
        log.warning("file_missing", path=str(path), sample_id=sample_id)
        raise FileNotFoundError(f"Feature file not found: {path}")

    try:
        return pd.read_parquet(path)
    except Exception as e:
        log.error("single_feature_load_failed", path=str(path), error=str(e))
        raise


def load_sample_metadata(
    sample_id: str,
    results_dir: str | Path,
) -> dict:
    """Load the metadata.parquet for a sample -> dict of QC values."""
    try:
        df = load_sample_feature(sample_id, ".metadata.parquet", results_dir)
        if df.empty:
            log.warning("empty_metadata", sample_id=sample_id)
            return {}
        return df.iloc[0].to_dict()
    except Exception as e:
        log.error("metadata_fetch_failed", sample_id=sample_id, error=str(e))
        return {}

In [ ]:
# | export
def make_variant_key(df: pd.DataFrame) -> pd.Series:
    """Create a hashable variant key from genomic coordinates."""
    if df.empty:
        log.warning("empty_df_in_variant_key")
        return pd.Series(dtype=object)

    for col in VARIANT_KEY_COLS:
        if col not in df.columns:
            log.error("missing_variant_key_col", col=col)
            raise KeyError(f"Missing required col for variant key: {col}")

    return df[VARIANT_KEY_COLS].apply(
        lambda row: (
            row["Chromosome"],
            int(row["Start_Position"]),
            int(row["End_Position"]),
            row["Reference_Allele"],
            row["Tumor_Seq_Allele2"],
        ),
        axis=1,
    )


@dataclass
class EvalRun:
    """Records the exact conditions of an evaluation run."""

    run_id: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    n_total_labeled: int = 0
    n_evaluated_samples: int = 0
    features_evaluated: list[str] = field(default_factory=list)
    label_config: LabelConfig = field(default_factory=LabelConfig)

    def to_json(self, path: str | Path):
        try:
            with open(path, "w") as f:
                data = self.__dict__.copy()
                data["label_config"] = self.label_config.__dict__
                json.dump(data, f, indent=2)
            log.info("eval_run_saved", path=str(path), run_id=self.run_id)
        except Exception as e:
            log.error("eval_run_save_failed", path=str(path), error=str(e))
            raise